### Question 1: From Prices to Returns

In [55]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
import warnings

warnings.filterwarnings("ignore")

# Load pre-processed data with date index

# Daily log returns
stock_daily_returns = pd.read_csv(
    "./processed_data/stock_log_ret_daily.csv", index_col=0
)
mkt_daily_returns = pd.read_csv("./processed_data/mkt_log_ret_daily.csv", index_col=0)

# Daily excess log returns
stock_excess_daily = pd.read_csv("./processed_data/stock_excess_daily.csv", index_col=0)

# Daily risk-free rate
rf_d = pd.read_csv("./processed_data/rf_daily.csv", index_col=0)["rf_daily"]

# Monthly excess log returns
stock_excess = pd.read_csv("./processed_data/stock_excess_monthly.csv", index_col=0)
mkt_excess = pd.read_csv("./processed_data/mkt_excess_monthly.csv", index_col=0)[
    "mkt_excess"
]
# Monthly risk-free rate
rf_m = pd.read_csv("./processed_data/rf_monthly.csv", index_col=0)["rf"]

### Task (a)

Compute daily log returns for each stock and for the market index


In [56]:
# Daily log returns for each stock and the market index were computed in the
# data pre-processing file, using the code:
#    log_ret = np.log(prices / prices.shift(1)).dropna()
#    mkt_ret = np.log(mkt_px / mkt_px.shift(1)).dropna()

# Combine them here.
log_returns = stock_daily_returns.copy()
log_returns["mkt_daily_returns"] = mkt_daily_returns

# Drop any months where the market return is missing.
log_returns = log_returns.dropna(subset=["mkt_daily_returns"])
log_returns.head()


,A,AAPL,ABBV,ABT,ACGL,ACN,ADBE,ADI,ADM,ADP,...,WY,WYNN,XEL,XOM,XYL,YUM,ZBH,ZBRA,ZTS,mkt_daily_returns
2015-03-05,0.005225,-0.016709,-0.058242,0.009106,0.009391,0.012462,0.012672,0.000856,0.008460,0.001962,...,-0.004094,0.001296,0.010071,-0.005060,-0.003087,0.005479,0.003530,0.001427,0.004963,0.001195
2015-03-06,-0.016479,0.001502,-0.021690,-0.020013,-0.000668,-0.015105,-0.013703,-0.008936,-0.026247,-0.008684,...,-0.032761,-0.014493,-0.037333,-0.012880,-0.011590,-0.017034,-0.024032,-0.016926,-0.014744,-0.014275
2015-03-09,0.005044,0.004256,-0.001799,0.007072,0.010633,0.001323,0.004888,0.009107,0.003022,0.009720,...,0.015323,-0.030836,0.012698,-0.005504,0.002272,0.003531,0.005144,-0.002569,0.011079,0.003937
2015-03-10,-0.026953,-0.020903,0.005387,-0.014194,-0.011803,-0.030531,-0.024946,-0.021789,-0.018932,-0.019653,...,-0.008985,-0.025387,0.002345,-0.010624,-0.015726,-0.019833,-0.012389,-0.018059,-0.012828,-0.017107
2015-03-11,0.005400,-0.018400,0.013872,0.002811,0.015926,-0.013721,0.000000,0.001572,-0.000439,0.003890,...,0.004503,-0.022489,-0.009117,-0.002853,-0.003464,-0.017878,0.006214,0.002275,0.006761,-0.001920


### Task (b)

Build the components needed for mean-variance optimization, annualized from daily data.

In [57]:
# u_i_hat = 252 * r_i_bar - r_f

# Calculate average daily log returns for each stock (r_i_bar)
avg_daily_ret = stock_daily_returns.mean()

# Calculate the average annualized risk-free rate (r_f)
# Since rf_d is already daily log, we scale the average of daily rf rates to annual.
avg_rf_annual = 252 * rf_d.mean()

# Compute Annualized Expected Excess Returns (u_i_hat)
mu_hat = (252 * avg_daily_ret) - avg_rf_annual

# Compute Annualized Covariance Matrix (Sigma)
# Formula: 252 * sample_covariance_of_daily_log_returns
sigma_daily = stock_daily_returns.cov()
sigma_hat = 252 * sigma_daily

print(f"Dimensions of mu_hat: {mu_hat.shape}")
print(f"Dimensions of sigma_hat: {sigma_hat.shape}")

Dimensions of mu_hat: (472,)
Dimensions of sigma_hat: (472, 472)


In [58]:
print(mu_hat)

A       0.091947
AAPL    0.187725
ABBV    0.141990
ABT     0.102747
ACGL    0.135193
          ...   
XYL     0.116206
YUM     0.096320
ZBH    -0.027413
ZBRA    0.094800
ZTS     0.113939
Length: 472, dtype: float64


In [59]:
print(sigma_hat)

             A      AAPL      ABBV       ABT      ACGL       ACN      ADBE  \
A     0.071653  0.036254  0.026105  0.036322  0.026454  0.037290  0.044051   
AAPL  0.036254  0.081182  0.021393  0.030257  0.025175  0.039007  0.054738   
ABBV  0.026105  0.021393  0.072099  0.027642  0.020766  0.021777  0.024135   
ABT   0.036322  0.030257  0.027642  0.056557  0.023983  0.030888  0.035554   
ACGL  0.026454  0.025175  0.020766  0.023983  0.072983  0.030788  0.025683   
...        ...       ...       ...       ...       ...       ...       ...   
XYL   0.038901  0.034445  0.021354  0.030085  0.036452  0.036929  0.036791   
YUM   0.025317  0.026515  0.017624  0.022687  0.026970  0.027519  0.028753   
ZBH   0.031575  0.027854  0.025366  0.030504  0.031315  0.031152  0.030094   
ZBRA  0.051608  0.052829  0.025454  0.036163  0.033292  0.046367  0.056216   
ZTS   0.038852  0.035230  0.027285  0.033780  0.025047  0.034585  0.040254   

           ADI       ADM       ADP  ...       WTW        WY    

### Task (c)

Pick 2–5 stocks that stand out statistically in some way—unusually high or low average returns, extreme volatility, anomalous correlations with the market, etc. For each, find news stories or firm-specific events that help explain the pattern you observe. Briefly discuss whether the statistical anomaly was foreseeable or only obvious in hindsight.

In [60]:
stocks = mu_hat.index

# Extreme average excess returns (mu_hat)
mu_mean = mu_hat.mean()
mu_std = mu_hat.std(ddof=1)
mu_z = (mu_hat - mu_mean) / mu_std

# Volatility (annualized)
vol_hat = np.sqrt(np.diag(sigma_hat))
vol_hat = pd.Series(vol_hat, index=stocks)
vol_mean = vol_hat.mean()
vol_std = vol_hat.std(ddof=1)
vol_z = (vol_hat - vol_mean) / vol_std

# Correlation with market (using daily log returns)
# Use the raw daily log returns and market daily returns
stock_daily_aligned = stock_daily_returns.loc[log_returns.index, stocks]
market_daily_aligned = log_returns["mkt_daily_returns"]

corr_with_mkt = stock_daily_aligned.corrwith(market_daily_aligned)

corr_mean = corr_with_mkt.mean()
corr_std = corr_with_mkt.std(ddof=1)
corr_z = (corr_with_mkt - corr_mean) / corr_std

summary = pd.DataFrame(
    {
        "mu_hat": mu_hat,
        "mu_z": mu_z,
        "vol_hat": vol_hat,
        "vol_z": vol_z,
        "corr_with_mkt": corr_with_mkt,
        "corr_z": corr_z,
    }
)

n = 5
print("Top and bottom stocks by annualized expected excess return (mu_hat):")
print("\nHighest mu_z:")
print(summary.sort_values("mu_z", ascending=False).head(n))
print("\nLowest mu_z:")
print(summary.sort_values("mu_z", ascending=True).head(n))

print("\nTop and bottom stocks by annualized volatility (vol_hat):")
print("\nHighest vol_z:")
print(summary.sort_values("vol_z", ascending=False).head(n))
print("\nLowest vol_z:")
print(summary.sort_values("vol_z", ascending=True).head(n))

print("\nTop and bottom stocks by correlation with the market:")
print("\nHighest corr_z:")
print(summary.sort_values("corr_z", ascending=False).head(n))
print("\nLowest corr_z:")
print(summary.sort_values("corr_z", ascending=True).head(n))

Top and bottom stocks by annualized expected excess return (mu_hat):

Highest mu_z:
        mu_hat      mu_z   vol_hat     vol_z  corr_with_mkt    corr_z
NVDA  0.516981  5.482562  0.488036  2.068169       0.645086  0.710027
AMD   0.328116  3.112939  0.576250  3.116018       0.513298 -0.501492
TPL   0.326127  3.087987  0.451004  1.628286       0.402007 -1.524573
AXON  0.292992  2.672251  0.478128  1.950484       0.435546 -1.216254
FICO  0.287330  2.601217  0.342829  0.343343       0.662635  0.871344

Lowest mu_z:
        mu_hat      mu_z   vol_hat     vol_z  corr_with_mkt    corr_z
WBA  -0.192059 -3.413505  0.353157  0.466029       0.410826 -1.443501
VTRS -0.189236 -3.378089  0.387254  0.871046       0.411505 -1.437256
PARA -0.175279 -3.202970  0.466043  1.806928       0.425796 -1.305888
PCG  -0.136651 -2.718319  0.605096  3.458658       0.262276 -2.809107
WBD  -0.135252 -2.700775  0.452724  1.648722       0.405773 -1.489952

Top and bottom stocks by annualized volatility (vol_hat):

Hi